# Lesson 3, Exercise 2: Strategic Pruning on Llama-3.2-1B - Impact of Method and Target

**Goal:**
The objective of this exercise is to explore the nuanced impact of different pruning strategies when applied to a large and powerful language model like Llama-3.2-1B. You will investigate how the choice of pruning *method* (e.g., magnitude-based vs. random) and pruning *target* (e.g., MLP layers vs. attention layers) affects the model's output quality and inference speed at similar sparsity levels, even before any fine-tuning.

## 2. Imports and Configuration

In [1]:
import os
import torch
import torch.nn.utils.prune as prune
from transformers import AutoModelForCausalLM, AutoTokenizer
import time
import copy # For creating fresh model copies
import pandas as pd

_local_llama = "/voc/shared/models/llama/Llama-3.2-1B"
MODEL_NAME = os.environ.get("UDACI_MODEL", _local_llama if os.path.isdir(_local_llama) else "unsloth/Llama-3.2-1B")
if os.path.isdir(MODEL_NAME):
    os.environ["HF_HUB_OFFLINE"] = "1"

PRUNING_AMOUNT = 0.3 # Target ~30% sparsity in selected layers
NUM_LAYERS_TO_TARGET_EXAMPLE = 2 # Target first N layers for simplicity in this example

PROMPT_TEXT = "The future of AI is"
MAX_NEW_TOKENS_PRUNING = 30
NUM_SPEED_RUNS = 3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# bf16 on GPUs that support it, fp16 on older GPUs; on CPU keep bf16 (safetensors are stored in bf16, so the
# weights are memory-mapped and each deepcopy costs ~2.5 GB instead of ~5 GB in fp32).
if torch.cuda.is_available():
    model_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    model_dtype = torch.bfloat16
print(f"Using device: {device}, Model dtype: {model_dtype}")

# Llama-3.2-1B layer names (from print(model)): model.layers.{i}.mlp.{gate_proj,up_proj,down_proj}
# and model.layers.{i}.self_attn.{q_proj,k_proj,v_proj,o_proj}
LLAMA_MLP_GATE_PROJ_TARGETS = [f"model.layers.{i}.mlp.gate_proj" for i in range(NUM_LAYERS_TO_TARGET_EXAMPLE)]
LLAMA_MLP_UP_PROJ_TARGETS = [f"model.layers.{i}.mlp.up_proj" for i in range(NUM_LAYERS_TO_TARGET_EXAMPLE)]
LLAMA_MLP_DOWN_PROJ_TARGETS = [f"model.layers.{i}.mlp.down_proj" for i in range(NUM_LAYERS_TO_TARGET_EXAMPLE)]

LLAMA_ATTN_Q_PROJ_TARGETS = [f"model.layers.{i}.self_attn.q_proj" for i in range(NUM_LAYERS_TO_TARGET_EXAMPLE)]
LLAMA_ATTN_K_PROJ_TARGETS = [f"model.layers.{i}.self_attn.k_proj" for i in range(NUM_LAYERS_TO_TARGET_EXAMPLE)]
LLAMA_ATTN_V_PROJ_TARGETS = [f"model.layers.{i}.self_attn.v_proj" for i in range(NUM_LAYERS_TO_TARGET_EXAMPLE)]
LLAMA_ATTN_O_PROJ_TARGETS = [f"model.layers.{i}.self_attn.o_proj" for i in range(NUM_LAYERS_TO_TARGET_EXAMPLE)]


Using device: cpu, Model dtype: torch.bfloat16


## 3. Helper Functions

In [2]:
def get_module_by_name(model, module_name_str):
    """Gets a module from a model using its string name."""
    # "model.layers.0.mlp.gate_proj" -> getattr chain; ModuleList indices are plain attributes ("0")
    names = module_name_str.split('.')
    module = model
    for name_part in names:
        module = getattr(module, name_part)
    return module

def calculate_sparsity(module):
    """Calculates sparsity of a module's weight if it exists."""
    if hasattr(module, 'weight') and module.weight is not None:
        w = module.weight
        fraction = (w == 0).sum().item() / w.numel()   # zeros / total
        return fraction
    return 0.0

def apply_pruning_to_layers(model, layer_names_list, amount, method='l1_unstructured'):
    """Applies global unstructured pruning to a list of specified layers."""
    parameters_to_prune_tuples = []
    valid_layer_names_pruned = []
    for name_str in layer_names_list:
        try:
            module = get_module_by_name(model, name_str)

            if module and hasattr(module, 'weight') and module.weight is not None:
                 parameters_to_prune_tuples.append((module, 'weight'))
                 valid_layer_names_pruned.append(name_str)
            else:
                print(f"Warning: Layer {name_str} has no 'weight' or weight is None. Skipping.")
        except AttributeError:
            print(f"Warning: Layer {name_str} not found in model. Make sure names are correct. Skipping.")

    if not parameters_to_prune_tuples:
        print("No valid parameters found to prune for the given layer names.")
        return [] # Return empty list if no layers were pruned

    # Global unstructured pruning: the `amount` fraction of the smallest-|w| (or random) weights across ALL
    # listed tensors is masked, so per-layer sparsity may differ slightly from `amount`.
    if method == 'l1_unstructured':
        pruning_method_class = prune.L1Unstructured
    elif method == 'random_unstructured':
        pruning_method_class = prune.RandomUnstructured
    else:
        raise ValueError(f"Unsupported pruning method: {method}")
    prune.global_unstructured(parameters_to_prune_tuples, pruning_method=pruning_method_class, amount=amount)

    # Make pruning permanent: fold weight_orig * weight_mask back into .weight and drop the forward hook
    for module, param_name in parameters_to_prune_tuples:
        prune.remove(module, param_name)

    print(f"Applied {method} pruning (amount {amount*100:.1f}%) to {len(valid_layer_names_pruned)} layers.")
    return valid_layer_names_pruned # Return names of layers actually processed

def measure_generation_speed_and_quality(model, tokenizer, prompt, max_new_tokens, num_runs):
    total_time = 0
    generated_text_sample = "Error: Generation did not run."
    input_ids = tokenizer(prompt, return_tensors="pt").to(model.device if hasattr(model, 'device') and model.device is not None else device)

    with torch.no_grad():
        for i in range(num_runs):
            current_device = model.device if hasattr(model, 'device') and model.device is not None else device
            if current_device.type == 'cuda':
                torch.cuda.synchronize(current_device)
            start_time = time.perf_counter()

            # Greedy decoding for deterministic, comparable timing / outputs
            outputs = model.generate(
                **input_ids,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

            if current_device.type == 'cuda':
                torch.cuda.synchronize(current_device)
            end_time = time.perf_counter()

            if i == 0 and outputs is not None:
                generated_text_sample = tokenizer.decode(outputs[0], skip_special_tokens=True)
            if outputs is not None:
                 total_time += (end_time - start_time)
            else:
                total_time = float('inf') # Indicate error
                break

    avg_time = total_time / num_runs if num_runs > 0 and total_time != float('inf') else float('nan')
    return avg_time, generated_text_sample


## 4. Load Original Model and Tokenizer (Baseline)

In [3]:
results_summary_list = []
original_model = None
tokenizer = None

print(f"Loading original model: {MODEL_NAME}...")
try:
    original_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=model_dtype).to(device).eval()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    if original_model is None or tokenizer is None:
        raise ValueError("Original model or tokenizer not loaded. Check TODOs.")

    print("Original model and tokenizer loaded.")
    # Verify the layer names we target actually exist
    for _n in LLAMA_MLP_GATE_PROJ_TARGETS[:1] + LLAMA_ATTN_Q_PROJ_TARGETS[:1]:
        print(f"  {_n}: {get_module_by_name(original_model, _n)}")

    # Warm-up then measure the unpruned model
    measure_generation_speed_and_quality(original_model, tokenizer, PROMPT_TEXT, 5, 1)
    original_avg_time, original_output = measure_generation_speed_and_quality(
        original_model, tokenizer, PROMPT_TEXT, MAX_NEW_TOKENS_PRUNING, NUM_SPEED_RUNS)

    results_summary_list.append({
        "Configuration": "Original",
        "Avg Sparsity Targeted (%)": 0,
        "Avg Inference Time (s)": f"{original_avg_time:.4f}",
        "Speed-up vs Original": "1.00x",
        "Output Sample": original_output
    })
    print(f"Original Model: Avg Time={original_avg_time:.4f}s, Output='{original_output[:100]}...'\n")

except Exception as e:
    print(f"CRITICAL ERROR loading original model {MODEL_NAME}: {e}")
    print("Ensure correct MODEL_NAME, HF_TOKEN (if needed), and sufficient resources.")
    original_model = None # Prevent further execution if base model fails


Loading original model: unsloth/Llama-3.2-1B...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=5) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original model and tokenizer loaded.
  model.layers.0.mlp.gate_proj: Linear(in_features=2048, out_features=8192, bias=False)
  model.layers.0.self_attn.q_proj: Linear(in_features=2048, out_features=2048, bias=False)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[transformers] Both `max_new_tokens` (=30) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=30) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=30) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original Model: Avg Time=6.3345s, Output='The future of AI is here. It’s not just a buzzword anymore. It’s a reality that’s changing the way w...'



## 5. Pruning Experiments

Helper function to run a single pruning experiment configuration:

In [4]:
def run_single_pruning_experiment(config_name, base_model_to_copy, target_layer_names, pruning_amt, prune_method):
    if base_model_to_copy is None: # Guard if original model loading failed
        print(f"Skipping {config_name} as base model is not available.")
        results_summary_list.append({"Configuration": config_name, "Error": "Base model not loaded"})
        return

    print(f"\n--- Running: {config_name} (Method: {prune_method}) ---")
    pruned_model_instance = None
    try:
        # Fresh, unpruned copy for every experiment
        pruned_model_instance = copy.deepcopy(base_model_to_copy)

        if pruned_model_instance is None:
            raise ValueError("Pruned model instance not created.")

        # Apply pruning to the target layers
        processed_layers = apply_pruning_to_layers(pruned_model_instance, target_layer_names, pruning_amt, method=prune_method)

        avg_sparsity_achieved = 0.0
        if processed_layers: # Only calculate if some layers were actually pruned
            sparsities = [calculate_sparsity(get_module_by_name(pruned_model_instance, n)) for n in processed_layers]
            avg_sparsity_achieved = 100.0 * sum(sparsities) / len(sparsities)

        print(f"Average sparsity in targeted layers: {avg_sparsity_achieved:.2f}%")

        # Speed + quality of the pruned copy
        avg_time, output = measure_generation_speed_and_quality(
            pruned_model_instance, tokenizer, PROMPT_TEXT, MAX_NEW_TOKENS_PRUNING, NUM_SPEED_RUNS)

        speed_up_val = original_avg_time / avg_time if avg_time > 0 and not pd.isna(original_avg_time) else float('nan')

        results_summary_list.append({
            "Configuration": config_name,
            "Avg Sparsity Targeted (%)": f"{avg_sparsity_achieved:.2f}",
            "Avg Inference Time (s)": f"{avg_time:.4f}",
            "Speed-up vs Original": f"{speed_up_val:.2f}x",
            "Output Sample": output
        })
        print(f"{config_name}: Avg Time={avg_time:.4f}s, Speed-up={speed_up_val:.2f}x, Output='{output[:100]}...'\n")

    except Exception as e:
        print(f"Error during {config_name}: {e}")
        results_summary_list.append({"Configuration": config_name, "Error": str(e)})
    finally:
        del pruned_model_instance # Important to free memory
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


### 5.1 Part A: Pruning Method Comparison

In [5]:
if original_model is not None: # Proceed only if base model loaded successfully
    print("\n--- Part A: Pruning Method Comparison (Targeting MLP Gate Projections) ---")
    part_a_targets = LLAMA_MLP_GATE_PROJ_TARGETS # Example, choose a consistent set

    run_single_pruning_experiment("A1: Magnitude (L1) pruning – MLP gate_proj",
                                  original_model, part_a_targets, PRUNING_AMOUNT, 'l1_unstructured')

    run_single_pruning_experiment("A2: Random pruning – MLP gate_proj",
                                  original_model, part_a_targets, PRUNING_AMOUNT, 'random_unstructured')
else:
    print("Skipping Part A due to original model loading failure.")



--- Part A: Pruning Method Comparison (Targeting MLP Gate Projections) ---

--- Running: A1: Magnitude (L1) pruning – MLP gate_proj (Method: l1_unstructured) ---


[transformers] Both `max_new_tokens` (=30) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Applied l1_unstructured pruning (amount 30.0%) to 2 layers.
Average sparsity in targeted layers: 30.00%


[transformers] Both `max_new_tokens` (=30) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=30) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=30) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Applied random_unstructured pruning (amount 30.0%) to 2 layers.
Average sparsity in targeted layers: 30.00%


[transformers] Both `max_new_tokens` (=30) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=30) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A2: Random pruning – MLP gate_proj: Avg Time=6.1816s, Speed-up=1.02x, Output='The future of AI is here. The future of AI is here. The future of AI is here. The future of AI is he...'



### 5.2 Part B: Pruning Target Comparison

In [6]:
if original_model is not None: # Proceed only if base model loaded successfully
    print("\n\n--- Part B: Pruning Target Comparison (All using Magnitude Pruning) ---")

    # All MLP projections of the first N layers
    mlp_layers_for_b = LLAMA_MLP_GATE_PROJ_TARGETS + LLAMA_MLP_UP_PROJ_TARGETS + LLAMA_MLP_DOWN_PROJ_TARGETS
    run_single_pruning_experiment("B1: Magnitude pruning – ALL MLP projections",
                                  original_model, mlp_layers_for_b, PRUNING_AMOUNT, 'l1_unstructured')

    # All attention projections of the first N layers
    attn_layers_for_b = (LLAMA_ATTN_Q_PROJ_TARGETS + LLAMA_ATTN_K_PROJ_TARGETS
                         + LLAMA_ATTN_V_PROJ_TARGETS + LLAMA_ATTN_O_PROJ_TARGETS)
    run_single_pruning_experiment("B2: Magnitude pruning – ALL attention projections",
                                  original_model, attn_layers_for_b, PRUNING_AMOUNT, 'l1_unstructured')
else:
    print("Skipping Part B due to original model loading failure.")




--- Part B: Pruning Target Comparison (All using Magnitude Pruning) ---

--- Running: B1: Magnitude pruning – ALL MLP projections (Method: l1_unstructured) ---


Applied l1_unstructured pruning (amount 30.0%) to 6 layers.


[transformers] Both `max_new_tokens` (=30) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Average sparsity in targeted layers: 30.00%


[transformers] Both `max_new_tokens` (=30) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=30) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Applied l1_unstructured pruning (amount 30.0%) to 8 layers.
Average sparsity in targeted layers: 30.80%


[transformers] Both `max_new_tokens` (=30) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=30) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


B2: Magnitude pruning – ALL attention projections: Avg Time=6.0248s, Speed-up=1.05x, Output='The future of AI is here. The future of AI is here. The future of AI is here. The future of AI is he...'



## 6. Display Final Results Summary

In [7]:
df_final_results = pd.DataFrame(results_summary_list)
pd.set_option("display.max_colwidth", 200)
print("\n\n--- Overall Pruning Experiment Results Summary ---")
print(df_final_results.drop(columns=["Output Sample"], errors="ignore").to_string())
print("\n--- Output samples ---")
for _, r in df_final_results.iterrows():
    print(f"[{r['Configuration']}]\n  {r.get('Output Sample', r.get('Error', ''))!r}")




--- Overall Pruning Experiment Results Summary ---
                                       Configuration Avg Sparsity Targeted (%) Avg Inference Time (s) Speed-up vs Original
0                                           Original                         0                 6.3345                1.00x
1         A1: Magnitude (L1) pruning – MLP gate_proj                     30.00                 6.6056                0.96x
2                 A2: Random pruning – MLP gate_proj                     30.00                 6.1816                1.02x
3        B1: Magnitude pruning – ALL MLP projections                     30.00                 6.5092                0.97x
4  B2: Magnitude pruning – ALL attention projections                     30.80                 6.0248                1.05x

--- Output samples ---
[Original]
  'The future of AI is here. It’s not just a buzzword anymore. It’s a reality that’s changing the way we live, work, and play. AI is'
[A1: Magnitude (L1) pruning – MLP gate_p

## 7. Analysis and Discussion

*Run on CPU (Intel i7-10610U, 4 cores), Llama‑3.2‑1B in bf16, 30 % global unstructured sparsity in the targeted layers of the first 2 decoder blocks, greedy decoding of 30 tokens, times averaged over 3 runs.*

| Configuration | Sparsity in targets | Avg time (s) | Speed‑up | Output |
|---|---|---|---|---|
| Original | 0 % | 6.33 | 1.00× | “…here. It’s not just a buzzword anymore. It’s a reality that’s changing the way we live, work, and play.” |
| A1 Magnitude – MLP `gate_proj` | 30.0 % | 6.61 | 0.96× | **identical to original** |
| A2 Random – MLP `gate_proj` | 30.0 % | 6.18 | 1.02× | degenerate loop “The future of AI is here.” ×N |
| B1 Magnitude – all MLP proj. | 30.0 % | 6.51 | 0.97× | degenerate loop |
| B2 Magnitude – all attention proj. | 30.8 % | 6.02 | 1.05× | degenerate loop |

1.  **Part A - Pruning Method:**
    *   At the same 30 % sparsity in the same two `gate_proj` matrices, **magnitude (L1) pruning left the greedy output token‑for‑token unchanged**, whereas **random pruning collapsed the model into a repetition loop** after the first sentence. Magnitude pruning removes the 30 % of weights with the smallest |w|, which contribute least to each pre‑activation, so the layer's function is barely perturbed; random pruning deletes large, important weights with the same probability as tiny ones and injects much more error into the residual stream. Speed was the same for both (see 4).

2.  **Part B - Pruning Target:**
    *   Pruning 30 % of *all* MLP projections (gate/up/down, 6 matrices ≈ 100 M weights) and 30 % of *all* attention projections (q/k/v/o, 8 matrices ≈ 17 M weights) in the first two blocks both broke the model — outputs became repetitive loops. Attention appears the more sensitive target: it has ~6× fewer parameters, so 30 % of it is a much smaller absolute number of weights, yet the damage was at least as bad. Attention projections are dense with structure (rotary‑embedded q/k, GQA‑shared k/v with only 8 heads), and early‑layer attention determines what later layers can see; the MLP is wider and more redundant, so it tolerated a single 30 % cut of `gate_proj` but not a cut of the whole block. Neither is safe at 30 % without recovery training.

3.  **Challenges with Llama-3.2-1B:**
    *   *Memory:* every experiment `deepcopy`s the model; in bf16 that is another ~2.5 GB, in fp32 it would be ~5 GB, and `torch.nn.utils.prune` temporarily holds `weight_orig` + a float mask per pruned tensor. On this 30 GB laptop we had to use bf16 and free the copy after each run. *Compute:* even 30‑token generations take ~6 s per run on CPU, so the five configurations × 3 runs take minutes. *Layer names:* the exercise required inspecting `print(model)` — Llama's modules are `model.layers.{i}.mlp.{gate,up,down}_proj` / `self_attn.{q,k,v,o}_proj`, unlike GPT‑2's `Conv1D` layers, and `getattr` must walk through the `ModuleList` index as a string.

4.  **Inference Speed-up:**
    *   **No.** All variants land within ±5 % of the original (6.0–6.6 s), i.e. within run‑to‑run noise on a shared CPU. Unstructured pruning only writes zeros into a dense tensor; the matmul kernels (oneDNN/BLAS on CPU, cuBLAS on GPU) still multiply every element, and after `prune.remove()` the tensor is exactly the same shape and dtype as before. A speed‑up requires either sparse kernels + a sparse storage format (and usually > 90 % sparsity before they beat dense BLAS), hardware‑supported semi‑structured 2:4 sparsity (Ampere+ GPUs), or *structured* pruning that removes whole neurons/heads and physically shrinks the matrices.

5.  **Necessity of Fine-tuning:**
    *   Only the mildest configuration (30 % of two `gate_proj` matrices) preserved output quality; every broader cut turned a fluent model into a two‑word repetition loop. In practice pruned LLMs are always followed by fine‑tuning (or use calibration‑aware one‑shot methods such as SparseGPT / Wanda) so the remaining weights can compensate for the removed ones — magnitude pruning alone tells you *where* the redundancy is, not how to exploit it without a recovery step.
